<a href="https://colab.research.google.com/github/GARVIT-bit/BIS/blob/main/Cuckoo_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import scipy.special as special

def objective_function(x):
    """
    The objective function to minimize.
    Using the classic Sphere function: f(x) = sum(x_i^2)
    """
    return np.sum(x**2)

def levy_flight(Lambda, size):
    """
    Generates step sizes from a Levy distribution using Mantegna's algorithm.
    """
    sigma1 = (special.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) /
             (special.gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2)))**(1 / Lambda)
    u = np.random.normal(0, sigma1, size)
    v = np.random.normal(0, 1, size)
    step = u / (np.abs(v)**(1 / Lambda))
    return step

def cuckoo_search(n_nests=25, max_iter=200, dim=10, pa=0.25, lb=-5.12, ub=5.12):
    """
    Parameters:
    n_nests  : Number of host nests (population size)
    max_iter : Maximum number of iterations
    dim      : Dimensionality of the problem
    pa       : Discovery rate of alien eggs/solutions
    lb, ub   : Lower and upper boundaries of search space
    """
    # Initialize host nests randomly within the boundaries
    nests = np.random.uniform(lb, ub, (n_nests, dim))

    # Evaluate the initial fitness of nests
    fitness = np.array([objective_function(nest) for nest in nests])

    # Find the current best nest
    best_idx = np.argmin(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    # Lambda parameter for Levy flight
    Lambda = 1.5

    for iteration in range(max_iter):
        # 1. Generate new solutions via Levy flights
        new_nests = nests.copy()
        for i in range(n_nests):
            step_size = levy_flight(Lambda, dim)
            # Update formula using Levy flight
            new_nests[i] += step_size * (nests[i] - best_nest) * np.random.normal(0, 1, dim)
            # Boundary control
            new_nests[i] = np.clip(new_nests[i], lb, ub)

            # Evaluate fitness of new cuckoo solution
            new_fit = objective_function(new_nests[i])

            # Randomly select a nest to compare
            random_nest_idx = np.random.randint(0, n_nests)
            if new_fit < fitness[random_nest_idx]:
                nests[random_nest_idx] = new_nests[i]
                fitness[random_nest_idx] = new_fit

        # 2. Abandon some nests with probability pa
        for i in range(n_nests):
            if np.random.rand() < pa:
                # Generate a completely new solution via a random step
                random_nest_idx1 = np.random.randint(0, n_nests)
                random_nest_idx2 = np.random.randint(0, n_nests)
                step_size = np.random.rand() * (nests[random_nest_idx1] - nests[random_nest_idx2])
                nests[i] += step_size
                nests[i] = np.clip(nests[i], lb, ub)
                fitness[i] = objective_function(nests[i])

        # Update the best solution found so far
        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        if (iteration + 1) % 20 == 0 or iteration == 0:
            print(f"Iteration {iteration+1:03d} | Best Fitness: {best_fitness:.6e}")

    return best_nest, best_fitness

# Run the algorithm
best_sol, best_val = cuckoo_search(n_nests=20, max_iter=150, dim=5)
print(f"\nOptimization Finished!")
print(f"Best Solution: {best_sol}")
print(f"Best Objective Value: {best_val:.6e}")

Iteration 001 | Best Fitness: 1.027926e+01
Iteration 020 | Best Fitness: 3.213978e-01
Iteration 040 | Best Fitness: 1.266233e-02
Iteration 060 | Best Fitness: 1.266233e-02
Iteration 080 | Best Fitness: 1.266233e-02
Iteration 100 | Best Fitness: 1.266233e-02
Iteration 120 | Best Fitness: 1.266233e-02
Iteration 140 | Best Fitness: 1.266233e-02

Optimization Finished!
Best Solution: [ 0.0278129  -0.10161632  0.01149878 -0.03780467 -0.00121805]
Best Objective Value: 1.266233e-02
